# Grupo X: preprocesamiento de Brent y riesgo geopolítico

## 1. Objetivo

El proyecto estudia si el riesgo geopolítico puede aportar información para predecir la variación mensual del precio del petróleo Brent. La pregunta guía es: ¿existe una relación entre el aumento del riesgo geopolítico y las variaciones mensuales del precio del Brent? Limpieza, unión por mes y generación el csv procesado. 

## 2. Importación de librerías

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent

BRENT_FILE = ROOT / "data" / "raw" / "brent_eia.xls"
GPR_FILE = ROOT / "data" / "raw" / "gpr_monthly.xls"
OUTPUT_FILE = ROOT / "data" / "processed" / "brent_gpr_monthly.csv"

## 3. Presentación de los datasets

- **Brent:** promedio mensual del precio spot del petróleo Brent, expresado en dólares por barril.
- **GPR:** índices mensuales de riesgo geopolítico. Para este proyecto se utilizan `GPR`, `GPRT` y `GPRA`.

## 4. Carga e inspección del dataset Brent

El archivo tiene las hojas `Contents` y `Data 1`. Los datos están en `Data 1` y los encabezados comienzan en la fila 3 de Excel, por eso usamos `header=2`.

In [2]:
print("Hojas del archivo Brent:", pd.ExcelFile(BRENT_FILE).sheet_names)

brent_raw = pd.read_excel(
    BRENT_FILE,
    sheet_name="Data 1",
    header=2,
)

display(brent_raw.head())
print("Dimensiones:", brent_raw.shape)
print()
print("Tipos de datos:")
print(brent_raw.dtypes)
print()
print("Período:", brent_raw["Date"].min(), "a", brent_raw["Date"].max())

Hojas del archivo Brent: ['Contents', 'Data 1']


,Date,Europe Brent Spot Price FOB (Dollars per Barrel)
0,1987-05-15,18.58
1,1987-06-15,18.86
2,1987-07-15,19.86
3,1987-08-15,18.98
4,1987-09-15,18.31


Dimensiones: (472, 2)

Tipos de datos:
Date                                                datetime64[us]
Europe Brent Spot Price FOB (Dollars per Barrel)           float64
dtype: object

Período: 1987-05-15 00:00:00 a 2026-08-15 00:00:00


Las columnas originales relevantes son `Date` y `Europe Brent Spot Price FOB (Dollars per Barrel)`. Las renombramos y llevamos la fecha al primer día de cada mes.

In [3]:
brent = brent_raw[[
    "Date",
    "Europe Brent Spot Price FOB (Dollars per Barrel)",
]].copy()

brent.columns = ["month", "brent_price"]
brent["month"] = pd.to_datetime(brent["month"], errors="coerce")
brent["brent_price"] = pd.to_numeric(brent["brent_price"], errors="coerce")
brent["month"] = brent["month"].dt.to_period("M").dt.to_timestamp()
brent = brent.sort_values("month").reset_index(drop=True)

display(brent.head())
print(brent.dtypes)

,month,brent_price
0,1987-05-01,18.58
1,1987-06-01,18.86
2,1987-07-01,19.86
3,1987-08-01,18.98
4,1987-09-01,18.31


month          datetime64[us]
brent_price           float64
dtype: object


## 5. Carga e inspección del dataset GPR

El archivo comienza en 1900 porque incluye índices históricos. El índice reciente `GPR` comienza en 1985. Se seleccionan solamente:

- `GPR`: riesgo geopolítico general.
- `GPRT`: amenazas geopolíticas.
- `GPRA`: actos geopolíticos efectivos.

No se utilizan `GPRH` ni los índices por país.

In [4]:
print("Hojas del archivo GPR:", pd.ExcelFile(GPR_FILE).sheet_names)

gpr_raw = pd.read_excel(GPR_FILE, sheet_name="Sheet1")

display(gpr_raw[["month", "GPR", "GPRT", "GPRA"]].head())
print("Dimensiones del archivo completo:", gpr_raw.shape)
print("Período del archivo completo:", gpr_raw["month"].min(), "a", gpr_raw["month"].max())
print("Inicio del GPR reciente:", gpr_raw.loc[gpr_raw["GPR"].notna(), "month"].min())

Hojas del archivo GPR:

 ['Sheet1']


,month,GPR,GPRT,GPRA
0,1900-01-01,NaN,NaN,NaN
1,1900-02-01,NaN,NaN,NaN
2,1900-03-01,NaN,NaN,NaN
3,1900-04-01,NaN,NaN,NaN
4,1900-05-01,NaN,NaN,NaN


Dimensiones del archivo completo: (1520, 115)
Período del archivo completo: 1900-01-01 00:00:00 a 2026-08-01 00:00:00
Inicio del GPR reciente: 1985-01-01 00:00:00


In [5]:
gpr = gpr_raw[["month", "GPR", "GPRT", "GPRA"]].copy()
gpr["month"] = pd.to_datetime(gpr["month"], errors="coerce")
gpr["month"] = gpr["month"].dt.to_period("M").dt.to_timestamp()

display(gpr.head())
print(gpr.dtypes)

,month,GPR,GPRT,GPRA
0,1900-01-01,NaN,NaN,NaN
1,1900-02-01,NaN,NaN,NaN
2,1900-03-01,NaN,NaN,NaN
3,1900-04-01,NaN,NaN,NaN
4,1900-05-01,NaN,NaN,NaN


month    datetime64[us]
GPR             float64
GPRT            float64
GPRA            float64
dtype: object


## 6. Unión mensual

Unimos ambos datasets por `month` y conservamos únicamente los meses presentes en los dos archivos. No reemplazamos valores faltantes con cero.

In [6]:
df = (
    brent.merge(gpr, on="month", how="inner")
    .sort_values("month")
    .reset_index(drop=True)
)

df = df[["month", "brent_price", "GPR", "GPRT", "GPRA"]]

print("Dimensiones después de la unión:", df.shape)
display(df.head())

Dimensiones después de la unión: (472, 5)


,month,brent_price,GPR,GPRT,GPRA
0,1987-05-01,18.58,98.650917,97.371407,95.235184
1,1987-06-01,18.86,94.182404,99.829254,81.829407
2,1987-07-01,19.86,98.871964,107.528946,88.264816
3,1987-08-01,18.98,99.968445,115.480316,87.601334
4,1987-09-01,18.31,121.167076,143.451981,97.539818


## 7. Precio del mes siguiente

La columna `brent_price_next_month` contiene el precio observado en el mes siguiente. Más adelante será la variable que se intentará predecir con información del mes actual.

In [7]:
df["brent_price_next_month"] = df["brent_price"].shift(-1)

if pd.isna(df.loc[df.index[-1], "brent_price_next_month"]):
    df = df.iloc[:-1].copy()

display(df.tail())

,month,brent_price,GPR,GPRT,GPRA,brent_price_next_month
466,2026-03-01,103.13,330.176819,319.203979,450.360352,117.29
467,2026-04-01,117.29,250.283661,268.601074,305.941193,107.14
468,2026-05-01,107.14,203.581985,233.492905,215.246140,85.40
469,2026-06-01,85.40,179.941696,214.942444,184.833160,83.76
470,2026-07-01,83.76,167.532242,190.894791,180.699921,91.08


## 8. Verificación final

Comprobamos el período, los faltantes y los meses duplicados antes de exportar.

In [8]:
assert df["brent_price_next_month"].notna().all()

print("Cantidad de observaciones:", len(df))
print("Período:", df["month"].min(), "a", df["month"].max())
print("Meses duplicados:", df["month"].duplicated().sum())
print()
print("Valores faltantes:")
display(df.isna().sum().to_frame("faltantes"))
print("Primeras filas:")
display(df.head())
print("Últimas filas:")
display(df.tail())

Cantidad de observaciones: 471
Período: 1987-05-01 00:00:00 a 2026-07-01 00:00:00
Meses duplicados: 0

Valores faltantes:


,faltantes
month,0
brent_price,0
GPR,0
GPRT,0
GPRA,0
brent_price_next_month,0


Primeras filas:


,month,brent_price,GPR,GPRT,GPRA,brent_price_next_month
0,1987-05-01,18.58,98.650917,97.371407,95.235184,18.86
1,1987-06-01,18.86,94.182404,99.829254,81.829407,19.86
2,1987-07-01,19.86,98.871964,107.528946,88.264816,18.98
3,1987-08-01,18.98,99.968445,115.480316,87.601334,18.31
4,1987-09-01,18.31,121.167076,143.451981,97.539818,18.76


Últimas filas:


,month,brent_price,GPR,GPRT,GPRA,brent_price_next_month
466,2026-03-01,103.13,330.176819,319.203979,450.360352,117.29
467,2026-04-01,117.29,250.283661,268.601074,305.941193,107.14
468,2026-05-01,107.14,203.581985,233.492905,215.246140,85.40
469,2026-06-01,85.40,179.941696,214.942444,184.833160,83.76
470,2026-07-01,83.76,167.532242,190.894791,180.699921,91.08


## 9. Exportación

Conservamos seis columnas y guardamos `month` con formato `YYYY-MM`.

In [9]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df_export = df[[
    "month",
    "brent_price",
    "GPR",
    "GPRT",
    "GPRA",
    "brent_price_next_month",
]].copy()

df_export["month"] = df_export["month"].dt.strftime("%Y-%m")
df_export.to_csv(OUTPUT_FILE, index=False)

print("Archivo guardado en:", OUTPUT_FILE)
print("Observaciones:", len(df_export))
print("Período:", df_export["month"].min(), "a", df_export["month"].max())
print("Valores faltantes:", int(df_export.isna().sum().sum()))
display(df_export.head())

Archivo guardado en: C:\Users\dra\Documents\brent-geopolitical-risk-predictor\data\processed\brent_gpr_monthly.csv
Observaciones: 471
Período: 1987-05 a 2026-07
Valores faltantes: 0


,month,brent_price,GPR,GPRT,GPRA,brent_price_next_month
0,1987-05,18.58,98.650917,97.371407,95.235184,18.86
1,1987-06,18.86,94.182404,99.829254,81.829407,19.86
2,1987-07,19.86,98.871964,107.528946,88.264816,18.98
3,1987-08,18.98,99.968445,115.480316,87.601334,18.31
4,1987-09,18.31,121.167076,143.451981,97.539818,18.76


# Incorporación de variables del mercado mundial de petróleo

Esta sección agrega producción, consumo y retiros netos de inventarios mundiales de la EIA. El procesamiento anterior de Brent y GPR se conserva sin cambios y `brent_gpr_monthly.csv` se utiliza solamente como entrada para la unión final.

## 1. Identificación del archivo EIA

El archivo encontrado es `3a._International_Petroleum_and_Other_Liquids_Production_Consumption_and_Inventories.csv`, una exportación de la **Table 3a** del STEO Data Browser. Al ser CSV no tiene hojas. Las primeras cuatro filas contienen metadatos y la fila 5 contiene los encabezados.

La estructura es horizontal: las primeras seis columnas describen cada serie y las columnas siguientes representan meses.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent

EIA_FILE = (
    ROOT
    / "data"
    / "raw"
    / "3a._International_Petroleum_and_Other_Liquids_Production_Consumption_and_Inventories.csv"
)
WORLD_OUTPUT_FILE = ROOT / "data" / "processed" / "world_oil_market_monthly.csv"
BRENT_GPR_FILE = ROOT / "data" / "processed" / "brent_gpr_monthly.csv"
FINAL_OUTPUT_FILE = ROOT / "data" / "processed" / "brent_gpr_market_monthly.csv"

with EIA_FILE.open(encoding="utf-8-sig") as file:
    metadata_eia = [next(file).strip() for _ in range(4)]

print("Metadatos del archivo:")
for line in metadata_eia:
    print(line)

eia_raw = pd.read_csv(EIA_FILE, skiprows=4)
print("Archivo utilizado:", EIA_FILE.name)
print("Dimensiones:", eia_raw.shape)
print("Columnas disponibles:", eia_raw.columns.tolist())

Metadatos del archivo:
"3a. International Petroleum and Other Liquids Production, Consumption, and Inventories"
"https://www.eia.gov/outlooks/steo/data/browser/?utm_source=chatgpt.com#/?v=6&f=M&s=0&start=199801&end=202712&maptype=0&ctype=linechart&id=&linechart=PAPR_WORLD"
"Tue Sep 22 2026 11:31:48 GMT-0300 (Argentina Standard Time)"
"Source: U.S. Energy Information Administration"
Archivo utilizado: 3a._International_Petroleum_and_Other_Liquids_Production_Consumption_and_Inventories.csv
Dimensiones: (40, 366)
Columnas disponibles: ['remove', 'Unnamed: 1', 'map', 'linechart', 'units', 'source key', 'Jan 1998', 'Feb 1998', 'Mar 1998', 'Apr 1998', 'May 1998', 'Jun 1998', 'Jul 1998', 'Aug 1998', 'Sep 1998', 'Oct 1998', 'Nov 1998', 'Dec 1998', 'Jan 1999', 'Feb 1999', 'Mar 1999', 'Apr 1999', 'May 1999', 'Jun 1999', 'Jul 1999', 'Aug 1999', 'Sep 1999', 'Oct 1999', 'Nov 1999', 'Dec 1999', 'Jan 2000', 'Feb 2000', 'Mar 2000', 'Apr 2000', 'May 2000', 'Jun 2000', 'Jul 2000', 'Aug 2000', 'Sep 2000'

Las series se identifican por `source key`, que evita depender de etiquetas repetidas. Se utilizan:

| Serie EIA | Variable final | Significado | Unidad |
|---|---|---|---|
| `PAPR_WORLD` | `world_production` | Producción mundial total de petróleo y otros líquidos | Millones de barriles por día |
| `PATC_WORLD` | `world_consumption` | Consumo mundial total de petróleo y otros líquidos | Millones de barriles por día |
| `T3_STCHANGE_WORLD` | `world_inventory_net_withdrawals` | Retiros netos mundiales de inventarios | Millones de barriles por día |

En retiros netos, un valor positivo representa una reducción de inventarios y uno negativo representa una acumulación.

In [2]:
series_keys = {
    "PAPR_WORLD": "world_production",
    "PATC_WORLD": "world_consumption",
    "T3_STCHANGE_WORLD": "world_inventory_net_withdrawals",
}

series_metadata = eia_raw.loc[
    eia_raw["source key"].isin(series_keys),
    ["remove", "Unnamed: 1", "units", "source key"],
]
display(series_metadata)

month_columns = list(eia_raw.columns[6:])
production_rows = eia_raw.loc[eia_raw["source key"] == "PAPR_WORLD", month_columns]

assert production_rows.nunique().max() == 1
print("Filas encontradas por serie:")
print(eia_raw["source key"].value_counts().reindex(series_keys).fillna(0).astype(int))
print("Las dos apariciones de PAPR_WORLD contienen la misma serie mensual.")

,remove,Unnamed: 1,units,source key
2,World Total,Total World Petroleum Production,million barrels per day,PAPR_WORLD
6,Total World Production,Total World Petroleum Production,million barrels per day,PAPR_WORLD
15,World Total,Total World Petroleum Consumption,million barrels per day,PATC_WORLD
31,World Total,World Total Crude Oil and Other Liquids Invent...,million barrels per day,T3_STCHANGE_WORLD


Filas encontradas por serie:
source key
PAPR_WORLD           2
PATC_WORLD           1
T3_STCHANGE_WORLD    1
Name: count, dtype: int64
Las dos apariciones de PAPR_WORLD contienen la misma serie mensual.


## 2. Validación de frecuencia

Los encabezados abarcan meses consecutivos desde enero de 1998 hasta diciembre de 2027. Por lo tanto, el archivo tiene frecuencia mensual y no hace falta repetir ni interpolar valores.

In [3]:
eia_months = pd.to_datetime(month_columns, format="%b %Y")
expected_months = pd.date_range(eia_months.min(), eia_months.max(), freq="MS")
missing_months_eia = expected_months.difference(eia_months)
expected_first_month = pd.Timestamp("1998-01-01")

print("Cantidad de meses:", len(eia_months))
print("Período completo:", eia_months.min(), "a", eia_months.max())
print("Meses faltantes en la secuencia:", len(missing_months_eia))
print("Comienza en enero de 1998:", eia_months.min() == expected_first_month)

if eia_months.min() != expected_first_month:
    raise ValueError(f"El archivo comienza en {eia_months.min():%Y-%m}, no en 1998-01.")

assert len(missing_months_eia) == 0

Cantidad de meses: 360
Período completo: 1998-01-01 00:00:00 a 2027-12-01 00:00:00
Meses faltantes en la secuencia: 0
Comienza en enero de 1998: True


## 3. Selección y limpieza de las series mundiales

Transformamos los meses de columnas a filas, convertimos fechas y valores, eliminamos duplicados exactos y ordenamos cronológicamente. No reemplazamos faltantes con cero.

In [4]:
selected_series = (
    eia_raw.loc[eia_raw["source key"].isin(series_keys)]
    .drop_duplicates(subset="source key", keep="first")
)

world_oil_full = (
    selected_series.set_index("source key")[month_columns]
    .T
    .rename(columns=series_keys)
    .rename_axis("month")
    .reset_index()
)

world_oil_full["month"] = pd.to_datetime(world_oil_full["month"], format="%b %Y")
numeric_columns = list(series_keys.values())
world_oil_full[numeric_columns] = world_oil_full[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce",
)
world_oil_full = (
    world_oil_full.drop_duplicates()
    .sort_values("month")
    .reset_index(drop=True)
)

print("Primeras filas:")
display(world_oil_full.head())
print("Últimas filas:")
display(world_oil_full.tail())
print("Cantidad de observaciones:", len(world_oil_full))
print("Período:", world_oil_full["month"].min(), "a", world_oil_full["month"].max())
print("Tipos de datos:")
print(world_oil_full.dtypes)
print("Faltantes:")
display(world_oil_full.isna().sum().to_frame("faltantes"))
print("Meses duplicados:", world_oil_full["month"].duplicated().sum())

Primeras filas:


source key,month,world_production,world_consumption,world_inventory_net_withdrawals
0,1998-01-01,76.49,73.08,-3.41
1,1998-02-01,76.90,74.50,-2.40
2,1998-03-01,76.71,74.30,-2.41
3,1998-04-01,76.60,72.76,-3.84
4,1998-05-01,75.97,70.74,-5.23


Últimas filas:


source key,month,world_production,world_consumption,world_inventory_net_withdrawals
355,2027-08-01,111.07,105.74,-5.33
356,2027-09-01,111.09,105.87,-5.22
357,2027-10-01,111.68,104.71,-6.97
358,2027-11-01,112.14,105.49,-6.65
359,2027-12-01,111.82,106.67,-5.15


Cantidad de observaciones: 360
Período: 1998-01-01 00:00:00 a 2027-12-01 00:00:00
Tipos de datos:
source key
month                              datetime64[us]
world_production                          float64
world_consumption                         float64
world_inventory_net_withdrawals           float64
dtype: object
Faltantes:


,faltantes
source key,
month,0
world_production,0
world_consumption,0
world_inventory_net_withdrawals,0


Meses duplicados: 0


## 4. Separación entre historia y pronóstico

El CSV no identifica cada mes como observado, estimado o pronosticado. Para que las variables mundiales queden alineadas con Brent y GPR, conservamos los datos de la EIA hasta el último mes disponible en `brent_gpr_monthly.csv`. Los valores de 2026 deben interpretarse como estimaciones preliminares del STEO, mientras que los meses posteriores al período compartido se excluyen.

Fuente de referencia: [U.S. EIA, Short-Term Energy Outlook, septiembre de 2026](https://www.eia.gov/outlooks/steo/report/).

In [5]:
brent_last_month = pd.to_datetime(
    pd.read_csv(BRENT_GPR_FILE, usecols=["month"])["month"],
    format="%Y-%m",
).max()
world_oil = world_oil_full.loc[world_oil_full["month"] <= brent_last_month].copy()
excluded_rows = world_oil_full.loc[world_oil_full["month"] > brent_last_month]

print("Período conservado:", world_oil["month"].min(), "a", world_oil["month"].max())
print("Observaciones conservadas:", len(world_oil))
print("Período posterior excluido:", excluded_rows["month"].min(), "a", excluded_rows["month"].max())
print("Observaciones posteriores excluidas:", len(excluded_rows))

Período conservado: 1998-01-01 00:00:00 a 2026-07-01 00:00:00
Observaciones conservadas: 343
Período posterior excluido: 2026-08-01 00:00:00 a 2027-12-01 00:00:00
Observaciones posteriores excluidas: 17


## 5. Balance mundial

Calculamos `world_market_balance` como producción menos consumo:

- Un resultado positivo indica que la producción supera al consumo.
- Un resultado negativo indica que el consumo supera a la producción.
- Es una aproximación del equilibrio entre oferta y demanda y no reemplaza una medición completa de inventarios.

La serie EIA de retiros netos tiene el signo opuesto: es positiva cuando se retiran existencias y negativa cuando se acumulan.

In [6]:
world_oil["world_market_balance"] = (
    world_oil["world_production"] - world_oil["world_consumption"]
)

inventory_check = (
    world_oil["world_market_balance"]
    + world_oil["world_inventory_net_withdrawals"]
).abs().max()

print("Máxima diferencia por redondeo entre balance y retiros netos:", round(inventory_check, 4))
display(world_oil.head())

Máxima diferencia por redondeo entre balance y retiros netos: 0.01


source key,month,world_production,world_consumption,world_inventory_net_withdrawals,world_market_balance
0,1998-01-01,76.49,73.08,-3.41,3.41
1,1998-02-01,76.90,74.50,-2.40,2.40
2,1998-03-01,76.71,74.30,-2.41,2.41
3,1998-04-01,76.60,72.76,-3.84,3.84
4,1998-05-01,75.97,70.74,-5.23,5.23


## 6. Verificación y exportación del dataset mundial

Validamos una observación por mes, una secuencia mensual completa y ausencia de faltantes antes de exportar.

In [7]:
expected_historical_months = pd.date_range(
    world_oil["month"].min(),
    world_oil["month"].max(),
    freq="MS",
)
historical_gaps = expected_historical_months.difference(world_oil["month"])

print("Cantidad de observaciones:", len(world_oil))
print("Fecha mínima:", world_oil["month"].min())
print("Fecha máxima:", world_oil["month"].max())
print("Meses duplicados:", world_oil["month"].duplicated().sum())
print("Saltos mensuales:", len(historical_gaps))
print("Faltantes:")
display(world_oil.isna().sum().to_frame("faltantes"))

assert world_oil["month"].min() == pd.Timestamp("1998-01-01")
assert not world_oil.isna().any().any()
assert not world_oil["month"].duplicated().any()
assert len(historical_gaps) == 0

world_export = world_oil[[
    "month",
    "world_production",
    "world_consumption",
    "world_inventory_net_withdrawals",
    "world_market_balance",
]].copy()
world_export["month"] = world_export["month"].dt.strftime("%Y-%m")
world_export.to_csv(WORLD_OUTPUT_FILE, index=False)

print("Archivo guardado en:", WORLD_OUTPUT_FILE)
display(world_export.head())

Cantidad de observaciones: 343
Fecha mínima: 1998-01-01 00:00:00
Fecha máxima: 2026-07-01 00:00:00
Meses duplicados: 0
Saltos mensuales: 0
Faltantes:


,faltantes
source key,
month,0
world_production,0
world_consumption,0
world_inventory_net_withdrawals,0
world_market_balance,0


Archivo guardado en: C:\Users\dra\Documents\brent-geopolitical-risk-predictor\data\processed\world_oil_market_monthly.csv


source key,month,world_production,world_consumption,world_inventory_net_withdrawals,world_market_balance
0,1998-01,76.49,73.08,-3.41,3.41
1,1998-02,76.90,74.50,-2.40,2.40
2,1998-03,76.71,74.30,-2.41,2.41
3,1998-04,76.60,72.76,-3.84,3.84
4,1998-05,75.97,70.74,-5.23,5.23


## 7. Unión con Brent y GPR

Cargamos el CSV existente de Brent-GPR sin modificarlo. Convertimos ambos meses a la misma referencia mensual y usamos un `inner merge`, por lo que solamente quedan los meses presentes en los dos datasets.

In [8]:
brent_gpr = pd.read_csv(BRENT_GPR_FILE)
world_monthly = pd.read_csv(WORLD_OUTPUT_FILE)

brent_gpr["month"] = pd.to_datetime(brent_gpr["month"], format="%Y-%m")
world_monthly["month"] = pd.to_datetime(world_monthly["month"], format="%Y-%m")

print("Brent-GPR:", len(brent_gpr), "filas,", brent_gpr["month"].min(), "a", brent_gpr["month"].max())
print("Mercado mundial:", len(world_monthly), "filas,", world_monthly["month"].min(), "a", world_monthly["month"].max())

brent_without_eia = pd.DatetimeIndex(
    sorted(set(brent_gpr["month"]) - set(world_monthly["month"]))
)
eia_without_brent = pd.DatetimeIndex(
    sorted(set(world_monthly["month"]) - set(brent_gpr["month"]))
)

print("Meses Brent-GPR sin EIA:", len(brent_without_eia))
print("Primeros meses sin EIA:", brent_without_eia[:10].strftime("%Y-%m").tolist())
print("Últimos meses sin EIA:", brent_without_eia[-10:].strftime("%Y-%m").tolist())
print("Meses EIA sin Brent-GPR:", eia_without_brent.strftime("%Y-%m").tolist())

df_final = (
    brent_gpr.drop(columns="brent_price_next_month")
    .merge(world_monthly, on="month", how="inner")
    .sort_values("month")
    .reset_index(drop=True)
)

print("Filas después de la unión:", len(df_final))
print("Período unido:", df_final["month"].min(), "a", df_final["month"].max())

Brent-GPR: 471 filas, 1987-05-01 00:00:00 a 2026-07-01 00:00:00
Mercado mundial: 343 filas, 1998-01-01 00:00:00 a 2026-07-01 00:00:00
Meses Brent-GPR sin EIA: 128
Primeros meses sin EIA: ['1987-05', '1987-06', '1987-07', '1987-08', '1987-09', '1987-10', '1987-11', '1987-12', '1988-01', '1988-02']
Últimos meses sin EIA: ['1997-03', '1997-04', '1997-05', '1997-06', '1997-07', '1997-08', '1997-09', '1997-10', '1997-11', '1997-12']
Meses EIA sin Brent-GPR: []
Filas después de la unión: 343
Período unido: 1998-01-01 00:00:00 a 2026-07-01 00:00:00


## 8. Secuencia mensual y variable objetivo

Antes de volver a crear `brent_price_next_month`, comprobamos que cada registro esté seguido por el mes calendario inmediato. Si hubiera saltos, el proceso se detendría para no confundir el siguiente registro disponible con el mes siguiente.

In [9]:
next_recorded_month = df_final["month"].shift(-1)
expected_next_month = df_final["month"] + pd.offsets.MonthBegin(1)
sequence_gaps = df_final.loc[
    next_recorded_month.notna() & next_recorded_month.ne(expected_next_month),
    ["month"],
]

print("Saltos en la secuencia mensual:", len(sequence_gaps))

if not sequence_gaps.empty:
    raise ValueError("Hay saltos mensuales; no se puede recalcular la variable objetivo.")

df_final["brent_price_next_month"] = df_final["brent_price"].shift(-1)

if pd.isna(df_final.loc[df_final.index[-1], "brent_price_next_month"]):
    df_final = df_final.iloc[:-1].copy()

assert df_final["brent_price_next_month"].notna().all()
print("Observaciones después de crear el objetivo:", len(df_final))
print("Período final:", df_final["month"].min(), "a", df_final["month"].max())

Saltos en la secuencia mensual: 0
Observaciones después de crear el objetivo: 342
Período final: 1998-01-01 00:00:00 a 2026-06-01 00:00:00


## 9. Verificación y exportación final

El dataset integrado conserva Brent, GPR, las variables mundiales y el precio real del Brent del mes siguiente. La exploración y el entrenamiento de modelos quedan para etapas posteriores.

In [10]:
final_columns = [
    "month",
    "brent_price",
    "GPR",
    "GPRT",
    "GPRA",
    "world_production",
    "world_consumption",
    "world_inventory_net_withdrawals",
    "world_market_balance",
    "brent_price_next_month",
]
df_final = df_final[final_columns]

final_expected_months = pd.date_range(
    df_final["month"].min(),
    df_final["month"].max(),
    freq="MS",
)
final_gaps = final_expected_months.difference(df_final["month"])

print("Cantidad de observaciones:", len(df_final))
print("Fecha mínima:", df_final["month"].min())
print("Fecha máxima:", df_final["month"].max())
print("Meses duplicados:", df_final["month"].duplicated().sum())
print("Saltos mensuales:", len(final_gaps))
print("Faltantes:")
display(df_final.isna().sum().to_frame("faltantes"))
print("Primeras filas:")
display(df_final.head())
print("Últimas filas:")
display(df_final.tail())

assert not df_final.isna().any().any()
assert not df_final["month"].duplicated().any()
assert len(final_gaps) == 0

final_export = df_final.copy()
final_export["month"] = final_export["month"].dt.strftime("%Y-%m")
final_export.to_csv(FINAL_OUTPUT_FILE, index=False)

print("Archivo guardado en:", FINAL_OUTPUT_FILE)

Cantidad de observaciones: 342
Fecha mínima: 1998-01-01 00:00:00
Fecha máxima: 2026-06-01 00:00:00
Meses duplicados: 0
Saltos mensuales: 0
Faltantes:


,faltantes
month,0
brent_price,0
GPR,0
GPRT,0
GPRA,0
world_production,0
world_consumption,0
world_inventory_net_withdrawals,0
world_market_balance,0
brent_price_next_month,0


Primeras filas:


,month,brent_price,GPR,GPRT,GPRA,world_production,world_consumption,world_inventory_net_withdrawals,world_market_balance,brent_price_next_month
0,1998-01-01,15.19,53.096302,59.821270,42.487816,76.49,73.08,-3.41,3.41,14.07
1,1998-02-01,14.07,77.597618,101.840828,45.368019,76.90,74.50,-2.40,2.40,13.10
2,1998-03-01,13.10,57.248062,65.501999,44.516712,76.71,74.30,-2.41,2.41,13.53
3,1998-04-01,13.53,50.996582,58.767384,38.002850,76.60,72.76,-3.84,3.84,14.36
4,1998-05-01,14.36,94.473488,129.798889,43.802757,75.97,70.74,-5.23,5.23,12.21


Últimas filas:


,month,brent_price,GPR,GPRT,GPRA,world_production,world_consumption,world_inventory_net_withdrawals,world_market_balance,brent_price_next_month
337,2026-02-01,70.89,119.865410,156.006012,81.240623,108.71,104.48,-4.23,4.23,103.13
338,2026-03-01,103.13,330.176819,319.203979,450.360352,96.90,102.14,5.24,-5.24,117.29
339,2026-04-01,117.29,250.283661,268.601074,305.941193,95.46,99.62,4.16,-4.16,107.14
340,2026-05-01,107.14,203.581985,233.492905,215.246140,94.41,99.17,4.76,-4.76,85.40
341,2026-06-01,85.40,179.941696,214.942444,184.833160,99.41,102.05,2.63,-2.64,83.76


Archivo guardado en: C:\Users\dra\Documents\brent-geopolitical-risk-predictor\data\processed\brent_gpr_market_monthly.csv
